# QTran: Three Semiconductor Datasets — 120-Epoch Training and Learning Curves

This notebook trains four parameter-matched models on **SECOM**, **Carinthia**, and **ST-AWFD D2** for exactly 120 epochs and plots Training Accuracy, Test Accuracy, Training Loss, and Test Loss.

Protocol safeguards:

- The split, seed, epoch count, optimiser family, and training budget are fixed before training.
- The test loader is monitored only for descriptive curves. It is never used for gradients, early stopping, checkpoint selection, hyperparameter selection, or choosing an epoch.
- All four models run for exactly 120 epochs.
- These curves are supplementary diagnostics; formal quantum-advantage claims must use the nested-CV summary and paired statistical tests.
- Times New Roman uses Chinese fifth-size type, i.e. **10.5 pt**.


## 1. Server preparation

Place the following licensed Windows font files in `/root/xxx/autodl/paper/fonts/` before running the notebook: `times.ttf`, `timesbd.ttf`, `timesi.ttf`, and `timesbi.ttf`.

Required dataset locations:

- `/root/xxx/autodl/data/raw/secom/`
- `/root/xxx/autodl/data_cache/carinthia_32.npz`
- `/root/xxx/autodl/data/raw/st_awfd_d2/D2.zip` (the loader also accepts `ST-AWFD_D2` or `D2` as the directory name)


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, Path('/root/xxx/autodl')]
PROJECT_ROOT = next((p for p in candidates if (p / 'qcs_core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate qcs_core.py. Open this notebook from /root/xxx/autodl or /root/xxx/autodl/paper.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'paper'))

from three_best_learning_curves import (
    choose_device, final_epoch_table, register_times_new_roman,
    run_all, save_figures,
)

print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
EPOCHS = 120
SEED = 42
RESUME = True
ALLOW_FONT_FALLBACK = False  # Keep False for final manuscript figures.
DEVICE = choose_device()

font_family = register_times_new_roman(
    PROJECT_ROOT / 'paper' / 'fonts',
    allow_fallback=ALLOW_FONT_FALLBACK,
)
print('Device:', DEVICE)
print('Figure font:', font_family)
print('Font size: 10.5 pt (Chinese fifth size)')


## 2. Train all 12 jobs

The experiment consists of 3 datasets × 4 models. Each completed job immediately saves `history.json` and `final.pt` under `artifacts/three_best_learning_curves/`. With `RESUME=True`, complete jobs are skipped when the cell is rerun. No formal nested-CV artifacts are overwritten.


In [ ]:
datasets, histories, parameter_table = run_all(
    project_root=PROJECT_ROOT,
    epochs=EPOCHS,
    seed=SEED,
    resume=RESUME,
    device=DEVICE,
)
display(parameter_table)


## 3. Plot and export publication figures

For each dataset, the notebook exports a 2×2 learning-curve figure. It also exports one combined 3×4 figure. PNG files use 600 dpi; PDF and SVG remain vector outputs.


In [ ]:
figure_paths = save_figures(PROJECT_ROOT, histories)
for path in figure_paths:
    print(path)


In [ ]:
final_table = final_epoch_table(histories)
table_path = PROJECT_ROOT / 'artifacts' / 'three_best_learning_curves' / 'final_epoch_metrics.csv'
table_path.parent.mkdir(parents=True, exist_ok=True)
final_table.to_csv(table_path, index=False)
display(final_table.sort_values(['dataset', 'test_accuracy'], ascending=[True, False]))
print('Saved:', table_path)


## Interpretation boundary

Do not select a model, epoch, or hyperparameter from the displayed test curves. Report epoch 120 because it was fixed in advance. A visually higher quantum curve is not by itself evidence of quantum advantage; the paper's principal evidence must remain multi-seed/fold test results, confidence intervals, effect sizes, and paired statistical comparisons.
